# Reward Hacking in Reinforcement Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/reward_hacking.ipynb)

An RL agent optimises the number you write down, not the outcome you meant. When
those drift apart you get **reward hacking**: the agent drives its score to the
ceiling while failing the real task. This notebook reproduces the failure in a 7x7
gridworld with a tabular Q-learning agent, then fixes it two ways and builds a
detector.

Companion post: [Reward Hacking in Reinforcement Learning](https://sesen.ai/blog/reward-hacking-reinforcement-learning).

## 1. The gridworld and a tabular Q-learning agent

The agent starts top-left; the goal is bottom-right and pays `+1` (and ends the
episode). A bonus tile in the centre pays `+1` **every time** the agent steps on it
and does not end anything: a well-meaning "helper" signal with a loophole. Three
reward modes share one true objective (reach the goal):

- `hackable`: true reward **+** the repeatable bonus (the loophole),
- `fixed`: the true reward only (reward the objective directly),
- `shaped`: the true reward **+** potential-based shaping (a safe helper).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right


class GridWorld:
    def __init__(self, size=7, bonus_pos=(3, 3), bonus=1.0, gamma=0.99,
                 reward_mode="hackable"):
        self.size, self.bonus_pos, self.bonus = size, bonus_pos, bonus
        self.gamma, self.reward_mode = gamma, reward_mode
        self.start, self.goal = (0, 0), (size - 1, size - 1)

    def reset(self):
        self.pos = self.start
        return self.pos

    def _phi(self, s):  # shaping potential: negative Manhattan distance to goal
        return -(abs(s[0] - self.goal[0]) + abs(s[1] - self.goal[1]))

    def step(self, a):
        dr, dc = ACTIONS[a]
        nr = min(max(self.pos[0] + dr, 0), self.size - 1)
        nc = min(max(self.pos[1] + dc, 0), self.size - 1)
        s_next = (nr, nc)
        done = s_next == self.goal
        r_true = 1.0 if done else 0.0                 # the objective
        if self.reward_mode == "hackable":
            r_proxy = r_true + (self.bonus if s_next == self.bonus_pos else 0.0)
        elif self.reward_mode == "fixed":
            r_proxy = r_true                          # proxy = objective
        elif self.reward_mode == "shaped":
            r_proxy = r_true + self.gamma * self._phi(s_next) - self._phi(self.pos)
        else:
            raise ValueError(self.reward_mode)
        self.pos = s_next
        return s_next, r_proxy, r_true, done

In [ ]:
def train(env, episodes=3000, max_steps=100, alpha=0.1,
          eps_start=1.0, eps_end=0.05, eps_decay_frac=0.6, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.size, env.size, 4))
    log = {"proxy": [], "true": [], "reached": []}
    for ep in range(episodes):
        eps = eps_end + (eps_start - eps_end) * max(0.0, 1 - ep / (episodes * eps_decay_frac))
        s = env.reset()
        g_proxy = g_true = 0.0
        disc = 1.0
        reached = False
        for _ in range(max_steps):
            a = rng.integers(4) if rng.random() < eps else int(np.argmax(Q[s]))
            s_next, r_proxy, r_true, done = env.step(a)
            target = r_proxy + env.gamma * (0.0 if done else np.max(Q[s_next]))
            Q[s][a] += alpha * (target - Q[s][a])
            g_proxy += disc * r_proxy
            g_true += disc * r_true
            disc *= env.gamma
            s = s_next
            if done:
                reached = True
                break
        log["proxy"].append(g_proxy)
        log["true"].append(g_true)
        log["reached"].append(reached)
    return Q, log


def rollout(env, Q, max_steps=100):
    s = env.reset()
    traj, visits, reached, bonus_hits = [s], np.zeros((env.size, env.size)), False, 0
    for _ in range(max_steps):
        s, _, _, done = env.step(int(np.argmax(Q[s])))
        traj.append(s); visits[s] += 1; bonus_hits += (s == env.bonus_pos)
        if done:
            reached = True
            break
    return traj, visits, reached, bonus_hits


def rolling(x, w=30):
    x = np.asarray(x, float)
    return np.convolve(x, np.ones(w) / w, mode="valid") if len(x) >= w else x

## 2. Watch the agent hack the reward

Train on the hackable reward and inspect the greedy policy. It never reaches the
goal; instead it shuttles on and off the bonus tile, collecting it dozens of times.

In [ ]:
env_hack = GridWorld(reward_mode="hackable")
Q_hack, log_hack = train(env_hack)
traj_hack, vis_hack, reached, bonus_hits = rollout(env_hack, Q_hack)
print(f"reached goal: {reached} | bonus tile grabbed: {bonus_hits} times "
      f"| rollout length: {len(traj_hack)}")
print(f"last-50-episode mean PROXY return: {np.mean(log_hack['proxy'][-50:]):.2f}")
print(f"last-50-episode mean TRUE  return: {np.mean(log_hack['true'][-50:]):.3f}")

The agent is not confused about the goal. With discount `gamma = 0.99`, reaching
the goal (12 steps away) is worth `gamma**12 ~= 0.89` once, while oscillating on the
bonus is worth ~50 over a 100-step episode. It correctly computed that ignoring the
goal pays better.

Logging the **proxy return** (what it optimises) against the **true return**
(discounted goal reward) shows the signature of hacking: the proxy soars, the true
return never leaves the floor.

In [ ]:
env_fix = GridWorld(reward_mode="fixed")
Q_fix, log_fix = train(env_fix)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
xr = np.arange(len(rolling(log_hack["proxy"])))
axes[0].plot(xr, rolling(log_hack["proxy"]), color="#d1495b", lw=2, label="Proxy return")
axes[0].plot(xr, rolling(log_hack["true"]), color="#0f8b8d", lw=2, label="True return")
axes[0].set_title("Hackable reward"); axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Discounted return"); axes[0].legend()
axes[1].plot(xr, rolling(log_fix["true"]), color="#0f8b8d", lw=3, label="True return")
axes[1].plot(xr, rolling(log_fix["proxy"]), color="#d1495b", lw=1.5, ls="--", label="Proxy (identical)")
axes[1].set_title("Fixed reward"); axes[1].set_xlabel("Episode"); axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Fix 1 — reward the objective directly

Drop the bonus and pay only for the goal. Proxy and true reward become the same
quantity, so there is nothing to game. Q-learning finds the optimal 12-step path.

In [ ]:
def evaluate(env, Q, n=50):
    reaches, steps = 0, []
    for _ in range(n):
        traj, _, reached, _ = rollout(env, Q)
        if reached:
            reaches += 1; steps.append(len(traj) - 1)
    return reaches / n, (np.mean(steps) if steps else None)

rate, steps = evaluate(env_fix, Q_fix)
print(f"fixed reward -> goal-reach rate: {rate:.2f} | mean steps-to-goal: {steps}")
print(f"gamma**12 = {0.99**12:.3f}  (optimal discounted return)")

## 4. Fix 2 — potential-based shaping

Sparse rewards can be slow, which is why the engineer reached for a bonus. The safe
way to add guidance is **potential-based shaping** (Ng, Harada & Russell, 1999):
add `F = gamma * Phi(s') - Phi(s)` for a state potential `Phi`. Summed over a
trajectory this telescopes to a constant, so the optimal policy is unchanged: you
cannot open a loophole. Only rewards aligned with the objective ever reach the goal.

In [ ]:
env_shaped = GridWorld(reward_mode="shaped")
Q_shaped, log_shaped = train(env_shaped)

w = 50
fig, ax = plt.subplots(figsize=(8, 4.2))
xr = np.arange(len(rolling(log_hack["reached"], w)))
ax.plot(xr, rolling(log_hack["reached"], w), color="#d1495b", lw=2, label="Hackable")
ax.plot(xr, rolling(log_fix["reached"], w), color="#0f8b8d", lw=2, label="Fixed")
ax.plot(xr, rolling(log_shaped["reached"], w), color="#8367c7", lw=2, ls="--", label="Shaped")
ax.set_xlim(0, 800); ax.set_ylim(-0.03, 1.05)
ax.set_title("Goal-reach rate by reward"); ax.set_xlabel("Episode")
ax.set_ylabel(f"Goal-reach rate (rolling {w})"); ax.legend()
plt.tight_layout(); plt.show()

print("shaped -> goal-reach rate:", evaluate(env_shaped, Q_shaped)[0])

## 5. A reward-hacking detector

You cannot detect hacking from the reward the agent optimises, because it always
goes up. You need an independent measure of the true objective. The simplest
detector is the gap between proxy and true return: when proxy climbs and true does
not follow, raise an alarm. (Caveat: potential-based shaping adds a constant offset,
so in practice watch whether the trusted metric is *improving*, not the raw gap.)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
gap_hack = rolling(np.array(log_hack["proxy"]) - np.array(log_hack["true"]))
gap_fix = rolling(np.array(log_fix["proxy"]) - np.array(log_fix["true"]))
xr = np.arange(len(gap_hack))
ax.plot(xr, gap_hack, color="#d1495b", lw=2, label="Hackable reward")
ax.plot(xr, gap_fix, color="#0f8b8d", lw=2, label="Fixed reward")
ax.axhline(1.0, color="#555", ls="--", lw=1.2, label="Alarm threshold")
ax.set_title("Detector: proxy return minus true return")
ax.set_xlabel("Episode"); ax.set_ylabel("Return gap"); ax.legend()
plt.tight_layout(); plt.show()

### Where the agent spends its time

The visitation heatmap makes the behaviour physical: the hacking agent camps on the
bonus tile; the fixed agent traces a path to the goal.

In [ ]:
_, vis_fix, _, _ = rollout(env_fix, Q_fix)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, vis, title in [(axes[0], vis_hack, "Hackable: camps on the bonus"),
                       (axes[1], vis_fix, "Fixed: walks to the goal")]:
    im = ax.imshow(vis, cmap="magma"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

## Exercises

1. **Close the loophole without removing the bonus.** Make the bonus one-shot
   (collectable once per episode) by tracking whether it has been taken. Does the
   hack disappear?
2. **Add a step penalty.** Give every step a small cost (e.g. `-0.05`) in the
   hackable reward. How large must it be before oscillating stops paying?
3. **Vary the discount.** Sweep `gamma` from 0.8 to 0.99 on the hackable reward. At
   what value does the agent stop hacking, and why?
4. **Design your own potential.** Replace the Manhattan-distance potential with a
   different `Phi(s)`. Confirm the agent still converges to the 12-step optimal path
   (policy invariance).
5. **Break the detector.** Construct a reward where proxy and true return both rise
   but the agent still misbehaves on a metric you did not log. What does this say
   about relying on a single trusted metric?

## References

- Amodei et al. (2016), *Concrete Problems in AI Safety*.
- Clark & Amodei (2016), *Faulty Reward Functions in the Wild* (OpenAI, CoastRunners).
- Ng, Harada & Russell (1999), *Policy Invariance under Reward Transformations*.
- Skalse et al. (2022), *Defining and Characterizing Reward Hacking*.
- Pan, Bhatia & Steinhardt (2022), *The Effects of Reward Misspecification*.